# Kapitel 9: Die Einbettungsschicht

> "Ein Wort erkennst du an seiner Gesellschaft." — **John Rupert Firth**, Linguist

---

## Was Sie lernen werden

- Warum Token-IDs allein nicht ausreichen und was Einbettungen tatsächlich lösen
- Wie man Token-Einbettungen von Grund auf mit Nachschlagetabellen erstellt
- Warum Positionsinformationen wichtig sind und wie man sie hinzufügt
- Wie man Token- und Positionseinbettungen zu vollständigen Eingaberepräsentationen kombiniert
- Wie man echte Modelleinbettungen erforscht und semantische Beziehungen entdeckt
- Praktische Initialisierungs- und Implementierungsüberlegungen

---

## Setup

Zuerst installieren wir die erforderlichen Pakete:

In [ ]:
# Erforderliche Pakete installieren
!pip install -q torch transformers

In [ ]:
# ===== IMPORTS =====
import torch                     # PyTorch: Tensor-Operationen und neuronale Netze
import torch.nn as nn            # Neuronale Netzwerk-Module (Schichten, etc.)
import torch.nn.functional as F  # Mathematische Funktionen (cosine_similarity, etc.)
from transformers import AutoModel, AutoTokenizer  # Vortrainierte Modelle

# Kurze Tensor-Erinnerung:
# - torch.tensor([1,2,3]) erstellt einen 1D-Tensor (wie eine Liste)
# - torch.randn(3, 4) erstellt einen 3×4-Tensor mit Zufallszahlen
# - tensor[0] indiziert in die erste Dimension
# - tensor.shape zeigt die Dimensionen

## 1. Warum können wir nicht einfach Token-IDs verwenden?

Token-IDs sind willkürliche Ganzzahlen ohne semantische Bedeutung. Sehen wir uns das Problem an:

In [ ]:
# Token-IDs sind nur Ganzzahlen - keine Beziehungen
token_ids = {
    "cat": 3797,
    "kitten": 28387,
    "dog": 4273,
    "car": 1097
}

print("Token-IDs:")
for word, id in token_ids.items():
    print(f"  '{word}' → {id}")

# Problem: "cat" ist numerisch näher an "dog" als an "kitten"!
print(f"\nDistanz von 'cat' zu 'dog': {abs(3797 - 4273)}")
print(f"Distanz von 'cat' zu 'kitten': {abs(3797 - 28387)}")
print("\nAber semantisch sind 'cat' und 'kitten' ähnlicher!")

### Die Lösung: Einbettungen

Wandeln Sie jede Token-ID in einen dichten Vektor um, der Bedeutung erfasst.

**Was ist `nn.Embedding`?**
- Eine Nachschlagetabelle mit `vocab_size` Zeilen und `embed_dim` Spalten
- Jede Zeile ist ein Vektor, der ein Token repräsentiert
- Eingabe: Token-ID (Ganzzahl) → Ausgabe: diese Zeile (Vektor)
- Stellen Sie es sich wie ein Dictionary vor: `{0: [0.1, 0.2, ...], 1: [0.5, -0.1, ...], ...}`

In [ ]:
# ===== Das Problem: Token-IDs =====
token_ids = torch.tensor([3797, 28387, 4273])  # cat, kitten, dog
print(f"Token-IDs Form: {token_ids.shape}")
print(f"Nur Ganzzahlen: {token_ids}")

# ===== Die Lösung: Einbettungen =====
vocab_size = 50257  # GPT-2 Vokabular
embed_dim = 768     # GPT-2 Einbettungsdimension

# Einbettungsschicht erstellen (das ist eine Nachschlagetabelle!)
embedding = nn.Embedding(vocab_size, embed_dim)

# Vektoren für unsere Token-IDs nachschlagen
token_vectors = embedding(token_ids)
print(f"\nToken-Einbettungen Form: {token_vectors.shape}")
print(f"Vektor des ersten Tokens (erste 10 Dim.): {token_vectors[0, :10]}")

# Jetzt ist jedes Token ein 768-dimensionaler Vektor, der Bedeutung erfassen kann!

## 2. Token-Einbettungen: Die Nachschlagetabelle

Bauen wir Token-Einbettungen von Grund auf, um den Mechanismus zu verstehen.

### Schritt 1: Die Einbettungsmatrix erstellen

In [ ]:
vocab_size = 50257  # GPT-2 Vokabulargröße
embed_dim = 768     # Einbettungsdimension

# Einbettungsmatrix erstellen: eine Zeile pro Token
embedding_matrix = torch.randn(vocab_size, embed_dim)

print(f"Einbettungsmatrix Form: {embedding_matrix.shape}")
print(f"\nEinbettung von Token 3797 (erste 10 Dim.): {embedding_matrix[3797, :10]}")

### Schritt 2: Mehrere Tokens nachschlagen

In [ ]:
token_ids = torch.tensor([464, 3797, 3332])  # "The cat sat"

# Manuelles Nachschlagen (was Einbettungsschichten intern machen)
embeddings = embedding_matrix[token_ids]

print(f"Eingabeform: {token_ids.shape}")       # torch.Size([3])
print(f"Ausgabeform: {embeddings.shape}")     # torch.Size([3, 768])

# Jede Token-ID → ihr 768-dimensionaler Vektor
print(f"\nEinbettung von Token 464 (erste 5 Dim.): {embeddings[0, :5]}")
print(f"Einbettung von Token 3797 (erste 5 Dim.): {embeddings[1, :5]}")
print(f"Einbettung von Token 3332 (erste 5 Dim.): {embeddings[2, :5]}")

### Schritt 3: Batches verarbeiten

In [ ]:
# Batch von 2 Sequenzen, jede mit 4 Tokens
token_ids_batch = torch.tensor([
    [464, 3797, 3332, 319],    # Sequenz 1: "The cat sat on"
    [314, 588, 4695, 345]      # Sequenz 2: "I will help you"
])

print(f"Batch-Form: {token_ids_batch.shape}")  # torch.Size([2, 4])

# Einbettungen für den gesamten Batch nachschlagen
embeddings_batch = embedding_matrix[token_ids_batch]

print(f"Einbettungen Form: {embeddings_batch.shape}")  # torch.Size([2, 4, 768])
print("\nForm-Transformation: (batch, seq) → (batch, seq, embed_dim)")

### Schritt 4: PyTorchs nn.Embedding verwenden

In [ ]:
class TokenEmbedding(nn.Module):
    """
    Token-Einbettungsschicht: wandelt Token-IDs in dichte Vektoren um.
    
    Das ist, was die 'wte' (word token embeddings) Schicht von GPT-2 macht.
    """
    def __init__(self, vocab_size, embed_dim):
        super().__init__()
        # Einbettungsmatrix als lernbarer Parameter erstellen
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        
        # Mit kleinen Zufallswerten initialisieren (GPT-2 Stil)
        nn.init.normal_(self.embedding.weight, mean=0.0, std=0.02)
    
    def forward(self, token_ids):
        """
        Args:
            token_ids: (batch, seq) Tensor von Token-IDs
        
        Returns:
            embeddings: (batch, seq, embed_dim) Tensor von Token-Vektoren
        """
        return self.embedding(token_ids)

# Token-Einbettungsschicht erstellen
token_embed = TokenEmbedding(vocab_size=50257, embed_dim=768)

# Einen Batch einbetten
token_ids = torch.tensor([[464, 3797, 3332, 319]])  # Form: (1, 4)
embeddings = token_embed(token_ids)

print(f"Eingabeform: {token_ids.shape}")        # torch.Size([1, 4])
print(f"Ausgabeform: {embeddings.shape}")      # torch.Size([1, 4, 768])
print(f"\nEinbettung des ersten Tokens (erste 5 Dim.): {embeddings[0, 0, :5]}")

## 3. Positionseinbettungen: Position lehren

Token-Einbettungen haben keinen Sinn für Position. Sehen wir uns das Problem an:

In [ ]:
# Zwei Sequenzen mit denselben Tokens, unterschiedlicher Reihenfolge
token_ids_1 = torch.tensor([[464, 3797, 3332]])  # "The cat sat"
token_ids_2 = torch.tensor([[3332, 3797, 464]])  # "sat cat The"

# Token-Einbettungen holen
token_embed = TokenEmbedding(vocab_size=50257, embed_dim=768)
embeddings_1 = token_embed(token_ids_1)
embeddings_2 = token_embed(token_ids_2)

print(f"Einbettungen 1 Form: {embeddings_1.shape}")
print(f"Einbettungen 2 Form: {embeddings_2.shape}")

# Sie sind unterschiedlich...
print(f"\nSind die Einbettungen identisch? {torch.equal(embeddings_1, embeddings_2)}")

# Aber wenn man beide sortiert, enthalten sie die gleichen Vektoren!
# Das ist das Problem: ohne Positionsinformation
# sehen "The cat sat" und "sat cat The" für Aufmerksamkeitsschichten gleich aus

### Gelernte Positionseinbettungen implementieren

In [ ]:
class PositionalEmbedding(nn.Module):
    """
    Gelernte Positionseinbettungen: ein trainierbarer Vektor pro Position.
    
    GPT-2 verwendet diesen Ansatz (genannt 'wpe' - word position embeddings).
    """
    def __init__(self, max_seq_len, embed_dim):
        """
        Args:
            max_seq_len: Maximale Sequenzlänge (z.B. 1024 für GPT-2)
            embed_dim: Einbettungsdimension (muss mit Token-Einbettungen übereinstimmen)
        """
        super().__init__()
        # Positionseinbettungsmatrix erstellen: (max_seq_len, embed_dim)
        self.pos_embed = nn.Embedding(max_seq_len, embed_dim)
        
        # Mit kleinen Zufallswerten initialisieren (GPT-2 Stil)
        nn.init.normal_(self.pos_embed.weight, mean=0.0, std=0.02)
        
        self.max_seq_len = max_seq_len
    
    def forward(self, token_ids):
        """
        Args:
            token_ids: (batch, seq) Tensor von Token-IDs
        
        Returns:
            pos_embeddings: (batch, seq, embed_dim) Tensor von Positionsvektoren
        """
        batch_size, seq_len = token_ids.shape
        
        # Sequenzlänge validieren
        if seq_len > self.max_seq_len:
            raise ValueError(
                f"Sequenzlänge {seq_len} überschreitet max_seq_len {self.max_seq_len}"
            )
        
        # Positionsindizes erstellen: [0, 1, 2, ..., seq_len-1]
        position_ids = torch.arange(
            seq_len,
            device=token_ids.device  # Gerät (CPU/GPU) der Eingabe angleichen
        )
        
        # Für Batch erweitern: (seq_len,) → (batch, seq_len)
        position_ids = position_ids.unsqueeze(0).expand(batch_size, seq_len)
        
        # Positionseinbettungen nachschlagen
        pos_embeddings = self.pos_embed(position_ids)
        
        return pos_embeddings

# Positionseinbettungsschicht erstellen
pos_embed = PositionalEmbedding(max_seq_len=1024, embed_dim=768)

# Beispiel: Sequenz der Länge 4
token_ids = torch.tensor([[464, 3797, 3332, 319]])  # Form: (1, 4)
pos_embeddings = pos_embed(token_ids)

print(f"Eingabeform: {token_ids.shape}")           # torch.Size([1, 4])
print(f"Positionseinbettungen Form: {pos_embeddings.shape}")  # torch.Size([1, 4, 768])

# Jede Position bekommt ihren eigenen gelernten Vektor
print(f"\nEinbettung Position 0 (erste 5 Dim.): {pos_embeddings[0, 0, :5]}")
print(f"Einbettung Position 1 (erste 5 Dim.): {pos_embeddings[0, 1, :5]}")
print(f"Einbettung Position 2 (erste 5 Dim.): {pos_embeddings[0, 2, :5]}")
print(f"Einbettung Position 3 (erste 5 Dim.): {pos_embeddings[0, 3, :5]}")

## 4. Token- + Positionseinbettungen kombinieren

### Die vollständige GPT2Embeddings-Klasse erstellen

In [ ]:
class GPT2Embeddings(nn.Module):
    """
    Vollständige GPT-2 Einbettungsschicht: Token- + Positionseinbettungen.
    
    Dies entspricht den 'wte' + 'wpe' Schichten von GPT-2.
    """
    def __init__(self, vocab_size, max_seq_len, embed_dim):
        """
        Args:
            vocab_size: Größe des Vokabulars (50257 für GPT-2)
            max_seq_len: Maximale Sequenzlänge (1024 für GPT-2)
            embed_dim: Einbettungsdimension (768 für GPT-2 Small)
        """
        super().__init__()
        
        # Token-Einbettungen: vocab_size × embed_dim
        self.token_embed = nn.Embedding(vocab_size, embed_dim)
        
        # Positionseinbettungen: max_seq_len × embed_dim
        self.pos_embed = nn.Embedding(max_seq_len, embed_dim)
        
        # Beide mit GPT-2s Standard-Initialisierung initialisieren
        nn.init.normal_(self.token_embed.weight, mean=0.0, std=0.02)
        nn.init.normal_(self.pos_embed.weight, mean=0.0, std=0.02)
        
        self.max_seq_len = max_seq_len
    
    def forward(self, token_ids):
        """
        Args:
            token_ids: (batch, seq) Tensor von Token-IDs
        
        Returns:
            embeddings: (batch, seq, embed_dim) Tensor kombinierter Einbettungen
        """
        batch_size, seq_len = token_ids.shape
        
        # Sequenzlänge validieren
        if seq_len > self.max_seq_len:
            raise ValueError(
                f"Sequenzlänge {seq_len} überschreitet max_seq_len {self.max_seq_len}"
            )
        
        # ===== Token-Einbettungen =====
        token_embeddings = self.token_embed(token_ids)
        
        # ===== Positionseinbettungen =====
        position_ids = torch.arange(seq_len, device=token_ids.device)
        position_ids = position_ids.unsqueeze(0).expand(batch_size, seq_len)
        position_embeddings = self.pos_embed(position_ids)
        
        # ===== Durch Addition kombinieren =====
        embeddings = token_embeddings + position_embeddings
        
        return embeddings

# GPT-2 Small Einbettungsschicht erstellen
gpt2_embed = GPT2Embeddings(
    vocab_size=50257,
    max_seq_len=1024,
    embed_dim=768
)

# Beispiel: einen Batch von Sequenzen einbetten (BEIDE müssen die gleiche Länge haben!)
# Kürzere Sequenzen werden mit 0en aufgefüllt, um zur längsten zu passen
token_ids = torch.tensor([
    [464, 3797, 3332, 319, 0, 0],  # "The cat sat on" + Auffüllung
    [314, 588, 4695, 345, 0, 0]    # "I will help you" + Auffüllung
])

embeddings = gpt2_embed(token_ids)

print(f"Eingabeform: {token_ids.shape}")         # torch.Size([2, 6])
print(f"Ausgabeform: {embeddings.shape}")       # torch.Size([2, 6, 768])
print(f"Jedes Token hat jetzt: {embeddings.shape[-1]} Dimensionen")

print(f"\nErste Sequenz, erstes Token (erste 10 Dim.):")
print(embeddings[0, 0, :10])

### Verbindung zu Kapitel 8: Die vollständige Pipeline

In [ ]:
from transformers import AutoTokenizer

# ===== Schritt 1: Tokenisieren (Kapitel 8) =====
tokenizer = AutoTokenizer.from_pretrained("gpt2")
text = "The cat sat on the mat"
token_ids = tokenizer.encode(text, return_tensors="pt")

print("Schritt 1: Tokenisierung")
print(f"Text: {text}")
print(f"Token-IDs: {token_ids}")
print(f"Form: {token_ids.shape}\n")

# ===== Schritt 2: Einbetten (Kapitel 9) =====
gpt2_embed = GPT2Embeddings(vocab_size=50257, max_seq_len=1024, embed_dim=768)
embeddings = gpt2_embed(token_ids)

print("Schritt 2: Einbettung")
print(f"Einbettungen Form: {embeddings.shape}")
print(f"Einbettung des ersten Tokens (erste 10 Dim.): {embeddings[0, 0, :10]}\n")

print("Schritt 3: Als Nächstes — Aufmerksamkeitsschichten (Kapitel 10)")
print(f"Diese {embeddings.shape} Einbettungen fließen in die Selbstaufmerksamkeit,")
print("wo Tokens aus dem Kontext der anderen lernen!")

## 5. GPT-2s Einbettungen erforschen

Laden wir ein echtes vortrainiertes GPT-2 Modell und erforschen, was es gelernt hat:

In [ ]:
# GPT-2 Small laden
model = AutoModel.from_pretrained("gpt2")
tokenizer = AutoTokenizer.from_pretrained("gpt2")

# Auf Token-Einbettungen zugreifen
token_embeddings = model.wte.weight  # Word Token Embeddings
print(f"Token-Einbettungen Form: {token_embeddings.shape}")
# torch.Size([50257, 768]) — ein 768-dim Vektor pro Token

# Auf Positionseinbettungen zugreifen
position_embeddings = model.wpe.weight  # Word Position Embeddings
print(f"Positionseinbettungen Form: {position_embeddings.shape}")
# torch.Size([1024, 768]) — ein 768-dim Vektor pro Position

### Ähnliche Wörter finden

**Was ist Kosinus-Ähnlichkeit?**
Misst, wie ähnlich zwei Vektoren basierend auf dem Winkel zwischen ihnen sind:
- **1.0** = identische Richtung (sehr ähnlich)
- **0.0** = senkrecht (unabhängig)
- **-1.0** = entgegengesetzte Richtung (gegenteilige Bedeutung)

Denken Sie daran als: "zeigen diese beiden Pfeile in die gleiche Richtung?"

In [ ]:
def find_similar_tokens(word, embeddings, tokenizer, top_k=5):
    """Finde Tokens mit Einbettungen, die dem gegebenen Wort am ähnlichsten sind."""
    # Token-ID für das Wort holen
    token_id = tokenizer.encode(word, add_special_tokens=False)[0]
    target_vec = embeddings[token_id]
    
    # Kosinus-Ähnlichkeit mit allen Tokens berechnen
    similarities = F.cosine_similarity(
        target_vec.unsqueeze(0),  # (1, 768)
        embeddings,               # (50257, 768)
        dim=1
    )
    
    # Top-k ähnlichste holen (ausgenommen das Wort selbst)
    top_indices = similarities.argsort(descending=True)[1:top_k+1]
    
    print(f"\nWörter, die '{word}' am ähnlichsten sind:")
    for idx in top_indices:
        token = tokenizer.decode([idx])
        score = similarities[idx].item()
        print(f"  {score:.3f} — '{token}'")

# Semantische Beziehungen erforschen
find_similar_tokens("king", token_embeddings, tokenizer, top_k=8)
find_similar_tokens("computer", token_embeddings, tokenizer, top_k=8)
find_similar_tokens("happy", token_embeddings, tokenizer, top_k=8)

### Probieren Sie Ihre eigenen Erkundungen aus!

In [ ]:
# Probieren Sie verschiedene Wörter aus:
words_to_explore = ["Python", "doctor", "fast", "beautiful"]

for word in words_to_explore:
    find_similar_tokens(word, token_embeddings, tokenizer, top_k=5)

### Bonus: Wortanalogien

Können wir Vektoren finden, die "king - man + woman ≈ queen" erfüllen?

In [ ]:
def word_analogy(word1, word2, word3, embeddings, tokenizer, top_k=5):
    """
    Finde: word1 - word2 + word3 ≈ ?
    Beispiel: king - man + woman ≈ queen
    """
    # Token-IDs holen
    id1 = tokenizer.encode(word1, add_special_tokens=False)[0]
    id2 = tokenizer.encode(word2, add_special_tokens=False)[0]
    id3 = tokenizer.encode(word3, add_special_tokens=False)[0]
    
    # Zielvektor berechnen: word1 - word2 + word3
    target_vec = embeddings[id1] - embeddings[id2] + embeddings[id3]
    
    # Ähnlichste Tokens finden
    similarities = F.cosine_similarity(
        target_vec.unsqueeze(0),
        embeddings,
        dim=1
    )
    
    # Eingabewörter aus den Ergebnissen ausschließen
    similarities[id1] = -1
    similarities[id2] = -1
    similarities[id3] = -1
    
    top_indices = similarities.argsort(descending=True)[:top_k]
    
    print(f"\n{word1} - {word2} + {word3} ≈ ?")
    for idx in top_indices:
        token = tokenizer.decode([idx])
        score = similarities[idx].item()
        print(f"  {score:.3f} — '{token}'")

# Klassisches Beispiel: king - man + woman ≈ queen
word_analogy("king", "man", "woman", token_embeddings, tokenizer, top_k=5)

# Probieren Sie andere aus!
word_analogy("Paris", "France", "Germany", token_embeddings, tokenizer, top_k=5)

## 6. Praktische Übungen

### Übung 1: Manuelles Einbettungs-Nachschlagen

In [ ]:
# Erstellen Sie ein kleines Vokabular (10 Tokens) und Einbettungsdimension von 8
# Erstellen Sie manuell eine Einbettungsmatrix und schlagen Sie Einbettungen für Token-IDs [2, 5, 7] nach
# Geben Sie die Formen bei jedem Schritt aus

# Schritt 1: Einbettungsmatrix erstellen
vocab_size = 10
embed_dim = 8
embedding_matrix = torch.randn(???, ???)  # Tragen Sie die Größen ein
print(f"Einbettungsmatrix Form: {embedding_matrix.shape}")

# Schritt 2: Token-IDs zum Nachschlagen erstellen
token_ids = torch.tensor([2, 5, 7])
print(f"Token-IDs: {token_ids}")

# Schritt 3: Einbettungen nachschlagen (Hinweis: verwenden Sie Indizierung wie embedding_matrix[...])
embeddings = ???
print(f"Einbettungen Form: {embeddings.shape}")

### Übung 2: TokenEmbedding von Grund auf erstellen

In [ ]:
# Implementieren Sie die TokenEmbedding-Klasse ohne auf das Beispiel zu schauen
# Fügen Sie ordnungsgemäße Initialisierung und Form-Validierung hinzu

# IHR CODE HIER

### Übung 3: Positionskodierung

In [ ]:
# Erstellen Sie eine Sequenz von Token-IDs: [10, 20, 30, 40, 50]
# Generieren Sie Positionsindizes und schlagen Sie sie in einer Positionseinbettungsschicht nach
# Überprüfen Sie, dass Position 0 immer den gleichen Vektor erhält, unabhängig vom Token

# IHR CODE HIER

### Übung 4: Vollständige GPT2Embeddings

In [ ]:
# Implementieren Sie die vollständige GPT2Embeddings-Klasse
# Testen Sie sie mit:
# - Einem Batch von 2 Sequenzen
# - Unterschiedlichen Sequenzlängen (eine Länge 5, eine Länge 8 mit Auffüllung)
# - Überprüfen Sie, dass die Ausgabeform (2, 8, embed_dim) ist

# IHR CODE HIER

### Übung 5: GPT-2 Ähnlichkeiten erforschen

In [ ]:
# Laden Sie GPT-2 und finden Sie Tokens ähnlich zu:
# - "Python" (sollte programmierbezogene Wörter finden)
# - "doctor" (sollte medizinische/berufliche Wörter finden)
# - "fast" (sollte geschwindigkeitsbezogene Wörter finden)
#
# Gruppiert das Modell verwandte Konzepte zusammen?

# IHR CODE HIER

### Übung 6: Geräte-Handhabung

In [ ]:
# Erstellen Sie eine GPT2Embeddings-Instanz
# Verschieben Sie sie auf GPU (falls verfügbar)
# Betten Sie Token-IDs ein, die auf CPU starten
# Was passiert? Beheben Sie es, indem Sie Token-IDs zuerst auf GPU verschieben

# IHR CODE HIER

### Übung 7: Sequenzlängen-Grenzen

In [ ]:
# Erstellen Sie eine GPT2Embeddings mit max_seq_len=10
# Versuchen Sie, eine Sequenz der Länge 15 einzubetten
# Behandeln Sie den Fehler elegant, indem Sie die Sequenz vor dem Einbetten kürzen

# IHR CODE HIER

### Übung 8: Integration mit Kapitel 8

In [ ]:
# Nehmen Sie die JSONL-Ausgabe aus Kapitel 8 (Ihren tokenisierten Datensatz)
# Laden Sie einen Datensatz, extrahieren Sie die Token-IDs, konvertieren Sie zu PyTorch-Tensor
# Betten Sie ihn mit GPT2Embeddings ein
# Geben Sie die Form bei jedem Schritt aus

# IHR CODE HIER

## Kapitelzusammenfassung

**Was wir gebaut haben:**

1. **Token-Einbettungen:** Nachschlagetabelle, die Token-IDs in semantische Vektoren umwandelt
2. **Positionseinbettungen:** Gelernte Vektoren, die Position in der Sequenz kodieren
3. **Vollständige GPT2Embeddings:** Kombiniert Token + Position durch Addition

**Was wir gelernt haben:**

- Token-IDs sind willkürliche Indizes ohne semantische Bedeutung
- Einbettungen wandeln IDs in dichte Vektoren um, die Beziehungen erfassen
- Positionsinformationen sind kritisch ("cat chased dog" ≠ "dog chased cat")
- GPT-2 verwendet gelernte Positionseinbettungen (trainierbare Parameter)
- Kosinus-Ähnlichkeit enthüllt gelernte semantische Beziehungen
- Echte Modelle lernen, dass "king" und "queen" verwandt sind, ohne es explizit gesagt zu bekommen!

**Als Nächstes:** Kapitel 10 wird diese Einbettungen für Selbstaufmerksamkeit verwenden, wo Tokens aus dem Kontext der anderen lernen!